# 01 - Explore the data
Goal: understand the simulator CSV before modelling. Run from the `ml/notebooks` folder with the `energyguard-ml` kernel.
If you have no CSV yet, the next cell makes a **stand-in** (`data/synthetic_30d.csv`). Replace it with Harsh's real export (`data/sim_30d.csv`) as soon as you have it.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
DATA_CANDIDATES = [Path("../data/sim_30d.csv"), Path("../data/synthetic_30d.csv")]
DATA = next((p for p in DATA_CANDIDATES if p.exists()), None)
print("using:", DATA)

In [ ]:
if DATA is None:
    from energyguard_ml.synthetic import make_synthetic
    Path("../data").mkdir(exist_ok=True)
    make_synthetic(days=30, meters=3, interval_s=60).to_csv("../data/synthetic_30d.csv", index=False)
    DATA = Path("../data/synthetic_30d.csv")
    print("made stand-in data ->", DATA)

In [ ]:
from energyguard_ml.training import load_csv
from energyguard_ml.features import APPLIANCES
df = load_csv(DATA)
print(df.shape)
print(df.columns.tolist())
df.head()

## Checks: labels, missing values, and whether total = sum of appliances

In [ ]:
print(df["sim_label"].value_counts())
print("\nshare of faulty rows: %.2f%%" % (100 * (df.sim_label != "normal").mean()))
print("\nnulls:\n", df.isna().sum()[lambda s: s > 0])
app_sum = df[[f"appliances_w.{a}" for a in APPLIANCES]].sum(axis=1)
print("\nmax |total - sum(appliances)|:", float((df.total_power_w - app_sum).abs().max()))
print("rows per meter:\n", df.groupby("meter_id").size())
print("median seconds between readings:", df.sort_values("timestamp").groupby("meter_id").timestamp.diff().dt.total_seconds().median())

## Daily pattern (timestamps are UTC) - this decides which hours count as 'quiet'

In [ ]:
normal = df[df.sim_label == "normal"].copy()
normal["hour"] = normal.timestamp.dt.hour
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
normal.groupby("hour").total_power_w.mean().plot(kind="bar", ax=ax[0], title="mean power by UTC hour (normal)")
normal.voltage_v.plot(kind="hist", bins=60, ax=ax[1], title="voltage, normal rows")
plt.tight_layout(); plt.show()
print(normal.voltage_v.describe())

## What do the faults look like?

In [ ]:
m = df[df.meter_id == df.meter_id.iloc[0]].sort_values("timestamp")
for kind in ["baseload_leakage", "high_load_spike", "voltage_fluctuation"]:
    idx = m.index[m.sim_label == kind]
    if len(idx) == 0:
        continue
    i = m.index.get_loc(idx[0])
    w = m.iloc[max(0, i - 30): i + 60]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3))
    ax[0].plot(w.timestamp, w.total_power_w); ax[0].set_title(f"{kind}: total_power_w")
    ax[1].plot(w.timestamp, w.voltage_v);    ax[1].set_title("voltage_v")
    for a in ax:
        a.axvspan(w[w.sim_label == kind].timestamp.min(), w[w.sim_label == kind].timestamp.max(), alpha=0.2, color="red")
    plt.tight_layout(); plt.show()